<a href="https://colab.research.google.com/github/ABINAYA600/GENAI_LAB/blob/main/program_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# EXPERIMENT 7
# FINE-TUNE PRE-TRAINED LANGUAGE MODELS FOR
# DOMAIN-SPECIFIC APPLICATIONS
# ============================================================

!pip install -q -U transformers datasets accelerate evaluate

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# ============================================================
# 1. DOMAIN-SPECIFIC DATASET
# ============================================================

data = {
    "text": [
        "The transformer model achieved excellent accuracy.",
        "Large Language Models are revolutionizing AI.",
        "The football team won the championship.",
        "The cricket match was exciting.",
        "Neural networks are widely used in deep learning.",
        "The player scored a brilliant goal.",
        "Machine learning improves decision making.",
        "The tennis tournament starts tomorrow."
    ],

    # 0 = Sports
    # 1 = Technology
    "label": [
        1, 1, 0, 0,
        1, 0, 1, 0
    ]
}

# Convert to Hugging Face Dataset
dataset = Dataset.from_dict(data)


# ============================================================
# 2. LOAD PRETRAINED TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)


# ============================================================
# 3. TOKENIZATION
# ============================================================

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )


dataset = dataset.map(
    tokenize
)

dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)


# ============================================================
# 4. LOAD PRETRAINED BERT MODEL
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)


# ============================================================
# 5. TRAINING CONFIGURATION
# ============================================================

training_args = TrainingArguments(
    output_dir="./fine_tuned_model",
    per_device_train_batch_size=2,
    num_train_epochs=2,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)


# ============================================================
# 6. FINE-TUNE MODEL
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

print("=" * 60)
print("STARTING MODEL FINE-TUNING")
print("=" * 60)

trainer.train()


# ============================================================
# 7. SAVE FINE-TUNED MODEL
# ============================================================

trainer.save_model(
    "./fine_tuned_model"
)

tokenizer.save_pretrained(
    "./fine_tuned_model"
)

print("\nModel saved successfully.")


# ============================================================
# 8. LOAD FINE-TUNED MODEL FOR INFERENCE
# ============================================================

fine_tuned_tokenizer = AutoTokenizer.from_pretrained(
    "./fine_tuned_model"
)

fine_tuned_model = AutoModelForSequenceClassification.from_pretrained(
    "./fine_tuned_model"
)


# ============================================================
# 9. PREDICTION FUNCTION
# ============================================================

def predict_class(text):

    inputs = fine_tuned_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    outputs = fine_tuned_model(
        **inputs
    )

    probabilities = outputs.logits.softmax(
        dim=-1
    )

    predicted_class = probabilities.argmax(
        dim=-1
    ).item()

    confidence = probabilities[0][
        predicted_class
    ].item()

    labels = {
        0: "Sports",
        1: "Technology"
    }

    return (
        labels[predicted_class],
        confidence
    )


# ============================================================
# 10. TEST NEW INPUT
# ============================================================

text = "Generative AI models improve intelligent automation."

predicted_class, confidence = predict_class(
    text
)


# ============================================================
# 11. DISPLAY RESULT
# ============================================================

print("\n")
print("=" * 60)
print("PREDICTION")
print("=" * 60)

print("Input :", text)
print("Predicted Class :", predicted_class)
print(
    "Confidence Score :",
    round(confidence, 3)
)


# ============================================================
# RESULT
# ============================================================

print("\n")
print("=" * 60)
print("RESULT")
print("=" * 60)

print(
    "Thus, a pretrained language model was successfully "
    "fine-tuned using domain-specific data and used to "
    "classify new text."
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.1 MB/s eta 0:00:00


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Step,Training Loss
1,0.654263
2,0.746866
3,0.727088
4,0.507207
5,0.493638
6,0.426569
7,0.502889
8,0.391982


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved successfully.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]



PREDICTION
Input : Generative AI models improve intelligent automation.
Predicted Class : Technology
Confidence Score : 0.644


RESULT
Thus, a pretrained language model was successfully fine-tuned using domain-specific data and used to classify new text.
